# Predicción final · Analista Sr · Lautaro Suarez
## Paso 2 de 2: ensemble de las 9 corridas de R1 y archivo de entrega

Este notebook **no entrena modelos**. Lee la predicción que dejó cada una de las 9 corridas del notebook
`PredFinal_Senior_Suarez_R1.ipynb` (una por semilla primigenia), promedia las probabilidades por cliente
y envía a todo cliente con probabilidad promedio mayor al umbral.

- Modelo: 9 corridas × 30 semillas de producción = 270 LightGBM, cada corrida con su propia optimización bayesiana.
- **Entrega: umbral 0,0325 → 9.714 envíos en 202109** (`ENS_R1x9_entrega_u0325_9714.csv`).
- También se genera la alternativa con umbral 0,030 (10.294 envíos), que no se entregó.
- El umbral se eligió en 202107 (local): 0,0325 rinde más que 0,030 en el ensemble (621,2 contra 619,1) y fue el que más veces ganó en un bootstrap de 500 remuestreos de clientes. El umbral se traslada solo a la cantidad de envíos de 202109.

Requisito: haber corrido antes `correr_9_semillas.sh`, que deja en cada `exp/WF<experimento>/` los archivos
`prediccion_local.txt` (202107) y `prediccion_entrega.txt` (202109).

In [1]:
require("data.table")

PARAM <- list()
PARAM$base <- path.expand("~/buckets/b1/exp")

# experimento (carpeta WF) = semilla primigenia de esa corrida de R1
PARAM$corridas <- c(
  WF20002 = 102191, WF20010 = 315779, WF20011 = 631861,
  WF20012 = 667657, WF20013 = 932227, WF20032 = 350377,
  WF20033 = 777767, WF20030 = 104729, WF20031 = 207013
)

PARAM$umbrales     <- c(entrega = 0.0325, alternativa = 0.030)
PARAM$gan_acierto  <- 117000
PARAM$gan_error    <- -3000
PARAM$cortes_local <- seq(8000, 14000, by = 500)
PARAM$carpeta_out  <- file.path(PARAM$base, "ENS", "FINAL")
PARAM$kaggle$competencia <- "labo-1-virtual-2026-analista-sr"
PARAM$kaggle$subir <- FALSE   # TRUE sube a Kaggle solo el archivo de entrega

Loading required package: data.table




Attaching package: ‘data.table’




The following object is masked from ‘package:base’:

    %notin%




In [2]:
# lee la probabilidad promedio de cada corrida y une por cliente
leer <- function(modo) {
  cols <- if (modo == "local") c("numero_de_cliente", "prob", "clase_ternaria") else c("numero_de_cliente", "prob")
  l <- lapply(names(PARAM$corridas), function(wf) {
    arch <- file.path(PARAM$base, wf, paste0("prediccion_", modo, ".txt"))
    if (!file.exists(arch)) stop("Falta ", arch, ": correr antes esa semilla con correr_9_semillas.sh")
    t <- fread(arch, select = cols)
    setnames(t, "prob", wf)
    t
  })
  Reduce(function(a, b) merge(a, b[, setdiff(names(b), "clase_ternaria"), with = FALSE], by = "numero_de_cliente"), l)
}

tb_local   <- leer("local")
tb_entrega <- leer("entrega")
wfs <- names(PARAM$corridas)
tb_local[,   prob := rowMeans(.SD), .SDcols = wfs]
tb_entrega[, prob := rowMeans(.SD), .SDcols = wfs]
cat("clientes 202107:", nrow(tb_local), "| clientes 202109:", nrow(tb_entrega), "\n")

clientes 202107: 164479 | clientes 202109: 165076 


In [3]:
# control en 202107 (local): ganancia de cada corrida, del ensemble y de los umbrales
gv <- ifelse(tb_local$clase_ternaria == "BAJA+2", PARAM$gan_acierto, PARAM$gan_error) / 120000
curva <- function(p) cumsum(gv[order(-p)])[PARAM$cortes_local]

res <- rbindlist(lapply(c(wfs, "prob"), function(n) {
  g <- curva(tb_local[[n]])
  data.table(modelo = ifelse(n == "prob", "ENSEMBLE", n), ganancia_max = max(g),
             corte_max = PARAM$cortes_local[which.max(g)])
}))
print(res)

for (u in PARAM$umbrales)
  cat(sprintf("umbral %.4f -> 202107: %d envios, ganancia %.1f | 202109: %d envios\n",
              u, sum(tb_local$prob > u), sum(gv[tb_local$prob > u]), sum(tb_entrega$prob > u)))

      modelo ganancia_max corte_max
      <char>        <num>     <num>
 1:  WF20002        624.0     12000
 2:  WF20010        613.0     12000
 3:  WF20011        613.0     10000
 4:  WF20012        615.5     13500
 5:  WF20013        623.5     11500
 6:  WF20032        611.5     12500
 7:  WF20033        616.5     10500
 8:  WF20030        625.5     11500
 9:  WF20031        623.0     12000
10: ENSEMBLE        621.0     13000


umbral 0.0325 -> 202107: 11312 envios, ganancia 621.2 | 202109: 9714 envios
umbral 0.0300 -> 202107: 11997 envios, ganancia 619.1 | 202109: 10294 envios


In [4]:
# archivos de entrega para 202109
dir.create(PARAM$carpeta_out, recursive = TRUE, showWarnings = FALSE)
for (nom in names(PARAM$umbrales)) {
  u <- PARAM$umbrales[[nom]]
  envios <- sum(tb_entrega$prob > u)
  arch <- file.path(PARAM$carpeta_out, sprintf("ENS_R1x9_%s_u%s_%d.csv", nom, sub("0\\.", "", sprintf("%.4f", u)), envios))
  fwrite(tb_entrega[, .(numero_de_cliente, Predicted = as.integer(prob > u))], arch)
  cat(nom, ":", arch, "-", envios, "envios\n")
  if (PARAM$kaggle$subir && nom == "entrega")
    system(paste0("kaggle competitions submit -c ", PARAM$kaggle$competencia, " -f ", arch,
                  " -m 'FINAL R1x9 ", nom, " umbral ", u, "'"))
}
fwrite(tb_entrega[, .(numero_de_cliente, prob)], file.path(PARAM$carpeta_out, "prediccion_entrega_ensemble.txt"), sep = "\t")

entrega : /home/ds/buckets/b1/exp/ENS/FINAL/ENS_R1x9_entrega_u0325_9714.csv - 9714 envios
alternativa : /home/ds/buckets/b1/exp/ENS/FINAL/ENS_R1x9_alternativa_u0300_10294.csv - 10294 envios
